In [3]:
# Import libraries
import pandas as pd
import glob

In [4]:
# Define raw data and output for processed data
RAW_DIR = "../data/raw"
OUTPUT_CSV = "../data/processed/london_crime_2023_2025_clean.csv"

In [5]:
# Sum total files in raw data folder
files = sorted(glob.glob(f"{RAW_DIR}/*.csv"))
print(f"{len(files)} files")

36 files


In [6]:
# Merge data
ref_cols = pd.read_csv(files[0], nrows=0).columns.tolist()
for f in files:
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    assert cols == ref_cols, f"Column mismatch in {f}:{cols}"
list_df = [pd.read_csv(f) for f in files]
df_merged = pd.concat(list_df, ignore_index=True)
print("Done")

Done


In [7]:
df_merged.shape

(3414776, 12)

In [8]:
df_merged.isnull().sum()

Crime ID                  690218
Month                          0
Reported by                    0
Falls within                   0
Longitude                  24260
Latitude                   24260
Location                       0
LSOA code                  24261
LSOA name                  24261
Crime type                     0
Last outcome category     690218
Context                  3414776
dtype: int64

In [9]:
df_merged = df_merged.drop(columns=["Context"])
df_merged.shape

(3414776, 11)

In [10]:
df_merged[df_merged["Crime ID"].isnull()]["Crime type"].value_counts()
(df_merged["Crime ID"].isnull() == df_merged["Last outcome category"].isnull()).all()

np.True_

In [11]:
df_merged["Month"] = pd.to_datetime(df_merged["Month"])

In [12]:
df_merged["Borough"] = df_merged["LSOA name"].str.extract(r"^(.*?)\s+\S+$")
df_merged["Borough"] = df_merged["Borough"].fillna("Unknown")
(df_merged["Borough"] == "Unknown").sum()
df_merged["Borough"].value_counts()

Borough
Westminster          301000
Camden               148321
Newham               146051
Southwark            141075
Tower Hamlets        139797
                      ...  
Taunton Deane             1
St Edmundsbury            1
West Somerset             1
Barrow-in-Furness         1
Ribble Valley             1
Name: count, Length: 351, dtype: int64

In [14]:
LONDON_BOROUGHS = [
    "Barking and Dagenham", "Barnet", "Bexley", "Brent", "Bromley", "Camden",
    "Croydon", "Ealing", "Enfield", "Greenwich", "Hackney",
    "Hammersmith and Fulham", "Haringey", "Harrow", "Havering", "Hillingdon",
    "Hounslow", "Islington", "Kensington and Chelsea", "Kingston upon Thames",
    "Lambeth", "Lewisham", "Merton", "Newham", "Redbridge",
    "Richmond upon Thames", "Southwark", "Sutton", "Tower Hamlets",
    "Waltham Forest", "Wandsworth", "Westminster", "City of London", "Unknown"
]

In [15]:
n_before = len(df_merged)
df_merged = df_merged[df_merged["Borough"].isin(LONDON_BOROUGHS)]
print(f"Dropped {n_before - len(df_merged)} rows outside the London boroughs "
      f"({(n_before - len(df_merged)) / n_before * 100:.2f}%)")

Dropped 13509 rows outside the London boroughs (0.40%)


In [16]:
has_id = df_merged["Crime ID"].notnull()
df_with_id = df_merged[has_id].drop_duplicates()
df_without_id = df_merged[~has_id]
df_clean = pd.concat([df_with_id, df_without_id], ignore_index=True)
print("Shape before/after true-duplicate removal:", df_merged.shape, "->", df_clean.shape)

Shape before/after true-duplicate removal: (3401267, 12) -> (3394911, 12)


In [17]:
df_clean.to_csv(OUTPUT_CSV, index=False)

In [18]:
import os
size_mb = os.path.getsize(OUTPUT_CSV) / (1024 * 1024)
check = pd.read_csv(OUTPUT_CSV)
print(f"Saved {OUTPUT_CSV} -- {size_mb:.1f} MB on disk, {check.shape[0]} rows read back")
assert check.shape[0] == df_clean.shape[0], "Row count mismatch after re-reading saved file!"
assert size_mb > 50, "File on disk is suspiciously small"
print("Borough sample values:", df_clean["Borough"].dropna().unique()[:10])
print("ALL CHECKS PASSED")

C:\Users\datph\AppData\Local\Temp\ipykernel_12000\2502569826.py:3: DtypeWarning: Columns (0: Crime ID, 1: Last outcome category) have mixed types. Specify dtype option on import or set low_memory=False.
  check = pd.read_csv(OUTPUT_CSV)


Saved ../data/processed/london_crime_2023_2025_clean.csv -- 813.6 MB on disk, 3394911 rows read back
Borough sample values: <ArrowStringArray>
['Barking and Dagenham',               'Barnet',               'Bexley',
                'Brent',              'Bromley',               'Camden',
       'City of London',              'Croydon',               'Ealing',
              'Enfield']
Length: 10, dtype: str
ALL CHECKS PASSED
